# Import libraries

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
from scipy.stats import qmc
from itertools import product
from scipy.spatial.distance import cdist
from scipy.stats.qmc import Sobol
import importlib
import CFDUtils
importlib.reload(CFDUtils)
from CFDUtils import meshGeneration

# Project directories

In [ ]:
currentDir = Path.cwd().resolve()

projectDir = next(
    (path for path in [currentDir, *currentDir.parents] if (path / '.git').exists()),
    None
)

openfoamProjectDir = Path(r'/path/to/OpenFOAM/run/airfoilCFDML')

# Experimental data

In [ ]:
expDataPath = projectDir/'CFD'/'data'/'expData.dat'
    
df = pd.read_csv(expDataPath, skiprows= 7, sep = r'\s+', header = None)
expData = df[:6].to_numpy(dtype=float)

# Mesh generation for mesh convergence

In [ ]:
# function to generate path for saving
def geoPathCreate(caseName):
    return projectDir/'gmsh'/'meshConvergence'/f'{caseName}'/'mesh.geo'

# mesh generation parameters
parameters = {
    'M' : 0,
    'P' : 0,
    'T' : 12,
    'meshLevels' : [1, 2, 3, 4], 
    'AoADeg' : 10.12, 
    'farfield' : 300, 
    'caseNames' : ['mesh1', 'mesh2', 'mesh3', 'mesh4']
}

# mesh generation
for I in range (4):
    M = parameters['M']
    P = parameters['P']
    T = parameters['T']
    meshLevel = parameters['meshLevels'][I]
    AoADeg = parameters['AoADeg']
    farfield = parameters['farfield']
    caseName = parameters['caseNames'][I]
    meshGeneration(M, P, T, meshLevel, AoADeg, farfield, caseName, geoPathCreate)

# Mesh generation for validation with mesh level 3

In [ ]:
# function to generate path for saving
def geoPathCreate (caseName):
    return projectDir/'gmsh'/'validationMesh3'/f'{caseName}'/'mesh.geo'

# mesh generation parameters
parameters = {
    'M' : 0,
    'P' : 0,
    'T' : 12,
    'meshLevel' : 3, 
    'AoADegs' : expData[:, 0], 
    'farfield' : 300, 
    'caseNames' : [f'n{-i:.2f}' if i<0 else f'{i:.2f}' for i in expData[:, 0]]
}
# mesh generation
for I in range (len(parameters['AoADegs'])):
    M = parameters['M']
    P = parameters['P']
    T = parameters['T']
    meshLevel = parameters['meshLevel']
    AoADeg = parameters['AoADegs'][I]
    farfield = parameters['farfield']
    caseName = parameters['caseNames'][I]
    meshGeneration(M, P, T, meshLevel, AoADeg, farfield, caseName, geoPathCreate)


# Mesh generation for validation with mesh level 4

In [ ]:
# function to generate path for saving
def geoPathCreate (caseName):
    return projectDir/'gmsh'/'validationMesh4'/f'{caseName}'/'mesh.geo'

# mesh generation parameters
parameters = {
    'M' : 0,
    'P' : 0,
    'T' : 12,
    'meshLevel' : 4, 
    'AoADegs' : expData[:, 0], 
    'farfield' : 300, 
    'caseNames' : [f'n{-i:.2f}' if i<0 else f'{i:.2f}' for i in expData[:, 0]]
}
# mesh generation
for I in range (len(parameters['AoADegs'])):
    M = parameters['M']
    P = parameters['P']
    T = parameters['T']
    meshLevel = parameters['meshLevel']
    AoADeg = parameters['AoADegs'][I]
    farfield = parameters['farfield']
    caseName = parameters['caseNames'][I]
    meshGeneration(M, P, T, meshLevel, AoADeg, farfield, caseName, geoPathCreate)


# ML training set generation

In [ ]:
# LHS generation information
camberedBounds = {
    'M' : [0.01, 6],
    'P' : [2, 6],
    'T' : [6, 24],
    'AoA' : [0, 8]
}

position = {
    'M' : 0,
    'P' : 1,
    'T' : 2,
    'AoA' : 3
}

# LHS generation parameters
d = 5  # points per dimension for boundary blocks

hypercubeParameters = {
    'cambered' : [300 - 2*d - 7*3*d - 12, 1],
    'symmetric' : [2*d, 2],
    'MFixedMax' : [3*d, 3],
    'PFixedMin' : [3*d, 4],
    'PFixedMax' : [3*d, 5],
    'TFixedMin' : [3*d, 6],
    'TFixedMax' : [3*d, 7],
    'AoAFixedMin' : [3*d, 8],
    'AoAFixedMax' : [3*d, 9]
}

# function to generate LHS samples
def latinHypercube(fixedParameter, fixedValue, randomSeed, nCases):
    dimensions = 4 - len(fixedParameter)
    
    lowerBounds = [camberedBounds[parameter][0] for parameter in camberedBounds if parameter not in fixedParameter]
    upperBounds = [camberedBounds[parameter][1] for parameter in camberedBounds if parameter not in fixedParameter]
    sampler = qmc.LatinHypercube(d = dimensions, optimization = 'random-cd', rng = np.random.default_rng(randomSeed))
    samples = qmc.scale(sampler.random(n=nCases), lowerBounds, upperBounds)
    for parameter in fixedParameter:
        samples = np.insert(samples, position[parameter], np.zeros(nCases)+fixedValue[fixedParameter.index(parameter)], axis = 1) 

    return samples

# generate training set
rawTrainingSet = np.vstack((
    np.column_stack((np.zeros((4, 1)), np.zeros((4, 1)), np.array(list(product(camberedBounds['T'], camberedBounds['AoA']))))),  # symmetric corner subset
    np.column_stack((np.zeros((8, 1))+6, np.array(list(product(camberedBounds['P'], camberedBounds['T'], camberedBounds['AoA']))))),  # maximum camber corner subset
    latinHypercube([], [], hypercubeParameters['cambered'][1], hypercubeParameters['cambered'][0]),  # cambered subset
    latinHypercube(['M', 'P'], [0, 0], hypercubeParameters['symmetric'][1], hypercubeParameters['symmetric'][0]),  # symmetric subset
    latinHypercube(['M'], [camberedBounds['M'][1]], hypercubeParameters['MFixedMax'][1], hypercubeParameters['MFixedMax'][0]),  # maximum camber boundary subset
    latinHypercube(['P'], [camberedBounds['P'][0]], hypercubeParameters['PFixedMin'][1], hypercubeParameters['PFixedMin'][0]),  # minimum camber position boundary subset
    latinHypercube(['P'], [camberedBounds['P'][1]], hypercubeParameters['PFixedMax'][1], hypercubeParameters['PFixedMax'][0]),  # maximum camber position boundary subset
    latinHypercube(['T'], [camberedBounds['T'][0]], hypercubeParameters['TFixedMin'][1], hypercubeParameters['TFixedMin'][0]),  # minimum thickness boundary subset
    latinHypercube(['T'], [camberedBounds['T'][1]], hypercubeParameters['TFixedMax'][1], hypercubeParameters['TFixedMax'][0]),  # maximum thickness boundary subset
    latinHypercube(['AoA'], [camberedBounds['AoA'][0]], hypercubeParameters['AoAFixedMin'][1], hypercubeParameters['AoAFixedMin'][0]),  # minimum angle of attack boundary subset
    latinHypercube(['AoA'], [camberedBounds['AoA'][1]], hypercubeParameters['AoAFixedMax'][1], hypercubeParameters['AoAFixedMax'][0])  # maximum angle of attack boundary subset

    ))

trainingSet = np.round(rawTrainingSet, 2)  # rounding to 2 dp

# checking whether rounding created identical cases
nBefore = rawTrainingSet.shape[0]
nUnique = np.unique(trainingSet, axis=0).shape[0]

print(f'Cases before rounding: {nBefore}')
print(f'Unique cases after rounding: {nUnique}')

# saving training set
trainingSet = pd.DataFrame(trainingSet, columns = ['M', 'P', 'T', 'AoA'])
trainingSet.insert(0, 'config', list(range(1, nUnique + 1)))

trainingSetPath = projectDir/'ML'/'data'/'trainingSet.txt'
trainingSetPath.parent.mkdir(parents=True, exist_ok=True)

trainingSet.to_csv(trainingSetPath, sep="\t", index=False)

# function to generate path for saving mesh
def geoPathCreate (caseName):
    return projectDir/'gmsh'/'ML'/f'{caseName}'/'mesh.geo'

# mesh generation parameters
parameters = {
    'Ms' : trainingSet['M'],
    'Ps' : trainingSet['P'],
    'Ts' : trainingSet['T'],
    'meshLevel' : 3, 
    'AoADegs' : trainingSet['AoA'], 
    'farfield' : 300, 
    'caseNames' : [f'{trainingSet['config'][i]}_{trainingSet['AoA'][i]}' for i in range (len(trainingSet))]
}
# mesh generation
for I in range (len(trainingSet)):
    M = parameters['Ms'][I]
    P = parameters['Ps'][I]
    T = parameters['Ts'][I]
    meshLevel = parameters['meshLevel']
    AoADeg = parameters['AoADegs'][I]
    farfield = parameters['farfield']
    caseName = parameters['caseNames'][I]
    meshGeneration(M, P, T, meshLevel, AoADeg, farfield, caseName, geoPathCreate)
    

# ML held-out test set generation

In [ ]:
# retrieve information on training set
trainingSet = pd.read_csv(trainingSetPath, sep = '\t').to_numpy()

isSymmetric = np.zeros((len(trainingSet), 1))  # add a binary toggle separate symmetric and cambered domain
isSymmetric[trainingSet[:, 1] == 0, 0] = 1
trainingSet = np.column_stack([trainingSet, isSymmetric])

trainingSet[trainingSet[:, 1] == 0, 2] = 4  # change the placeholder value of P to 4 when m = 0

normMinBounds = np.array([0, 2, 6, 0, 0])  # boundaries for normalising
normMaxBounds = np.array([6, 6, 24, 8, 1])

existingPoints = (trainingSet[:, 1:] - normMinBounds)/(normMaxBounds-normMinBounds)

# generate candidates in cambered domain
camberedSampler = Sobol(d = 4, seed = 1)
nCamberedCandidates = 2**16
camberedUnitCandidates = camberedSampler.random(nCamberedCandidates) 

MCandidates = np.round(0.01 + camberedUnitCandidates[:, 0] * (6 - 0.01), 2)
PCandidates = np.round(2.00 + camberedUnitCandidates[:, 1] * (6 - 2), 2)
TCandidates = np.round(6.00 + camberedUnitCandidates[:, 2] * (24 - 6), 2)
AoACandidates = np.round(0.00 + camberedUnitCandidates[:, 3] * (8 - 0), 2)

camberedCandidates = (np.column_stack([MCandidates, PCandidates, TCandidates, AoACandidates, np.zeros(nCamberedCandidates)]) - normMinBounds)/(normMaxBounds - normMinBounds)

# generate candidates in symmetric domain
symmetricSampler = Sobol(d = 2, seed = 1)
nsymmetricCandidates = 2**8
symmetricUnitCandidates = symmetricSampler.random(nsymmetricCandidates) 

TCandidates = np.round(6.00 + symmetricUnitCandidates[:, 0] * (24 - 6), 2)
AoACandidates = np.round(0.00 + symmetricUnitCandidates[:, 1] * (8 - 0), 2)

symmetricCandidates = (np.column_stack([np.zeros(nsymmetricCandidates), np.zeros(nsymmetricCandidates) + 4, TCandidates, AoACandidates, np.zeros(nsymmetricCandidates) + 1]) - normMinBounds)/(normMaxBounds - normMinBounds)

# select from candidates using greedy maximin infill
selectedNorm = []

nCamberedSelected = 40
nSymmetricSelected = 10

for i in range(nCamberedSelected):
    d = cdist(camberedCandidates, existingPoints)
    dMin = d.min(axis = 1)
    furthestIdx = np.argmax(dMin)

    selectedNorm.append(camberedCandidates[furthestIdx])
    existingPoints = np.vstack([existingPoints, camberedCandidates[furthestIdx]])

    camberedCandidates = np.delete(camberedCandidates, furthestIdx, axis = 0)

for i in range(nSymmetricSelected):
    d = cdist(symmetricCandidates, existingPoints)
    dMin = d.min(axis = 1)
    furthestIdx = np.argmax(dMin)

    selectedNorm.append(symmetricCandidates[furthestIdx])
    existingPoints = np.vstack([existingPoints, symmetricCandidates[furthestIdx]])

    symmetricCandidates = np.delete(symmetricCandidates, furthestIdx, axis = 0)

selected = np.round(np.array(selectedNorm) * (normMaxBounds - normMinBounds) + normMinBounds, 2)  # denormalising and rounding

# saving test set
testSet = pd.DataFrame(selected, columns = ['M', 'P', 'T', 'AoA', 'isSymmetric'])
testSet.insert(0, 'config', list(range(1, nCamberedSelected + nSymmetricSelected + 1)))

testSetPath = projectDir/'ML'/'data'/'testSet.txt'
testSetPath.parent.mkdir(parents=True, exist_ok=True)

testSet.to_csv(testSetPath, sep="\t", index=False)

# function to generate path for saving mesh
def geoPathCreate (caseName):
    return projectDir/'gmsh'/'MLTest'/f'{caseName}'/'mesh.geo'

# mesh generation parameters
parameters = {
    'Ms' : testSet['M'],
    'Ps' : testSet['P'],
    'Ts' : testSet['T'],
    'meshLevel' : 3, 
    'AoADegs' : testSet['AoA'], 
    'farfield' : 300, 
    'caseNames' : [f'{testSet['config'][i]}_{testSet['AoA'][i]}' for i in range (len(testSet))]
}

# mesh generation
for I in range (len(testSet)):
    M = parameters['Ms'][I]
    P = parameters['Ps'][I]
    T = parameters['Ts'][I]
    meshLevel = parameters['meshLevel']
    AoADeg = parameters['AoADegs'][I]
    farfield = parameters['farfield']
    caseName = parameters['caseNames'][I]
    meshGeneration(M, P, T, meshLevel, AoADeg, farfield, caseName, geoPathCreate)